# Step 1: Library Imports and Path Configuration

In [1]:
import os
import torch
from google.colab import drive
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split

# Mount Google Drive to access the extracted dataset from Day 1
drive.mount('/content/drive', force_remount=True)

# Define project and dataset paths
project_dir = "/content/drive/MyDrive/Brain_Tumor_Classification"
%cd {project_dir}

dataset_path = "dataset_extracted/Brain-Tumor-Classification-DataSet-master/Training"
print(f"Active Project Directory: {os.getcwd()}")

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1Rd5_epoDI2porCXmS9-4opjNV4Tdx9s2/Brain_Tumor_Classification
Active Project Directory: /content/drive/.shortcut-targets-by-id/1Rd5_epoDI2porCXmS9-4opjNV4Tdx9s2/Brain_Tumor_Classification


# Step 2: Image Transformation Pipeline and Train/Val/Test Splitting

In [2]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Define Image Transformations (Pipeline)
# ImageNet dimensions (224x224) and standard normalization statistics are utilized
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load the entire dataset using ImageFolder
full_dataset = datasets.ImageFolder(root=dataset_path, transform=data_transforms)
total_count = len(full_dataset)
print(f"Total images found for splitting: {total_count}")

# Calculate subset sizes (70% Train, 15% Validation, 15% Test)
train_size = int(0.70 * total_count)
val_size = int(0.15 * total_count)
test_size = total_count - train_size - val_size

# Perform the deterministic split
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

print(f"Train Set Size: {len(train_dataset)}")
print(f"Validation Set Size: {len(val_dataset)}")
print(f"Test Set Size: {len(test_dataset)}")

Total images found for splitting: 2870
Train Set Size: 2008
Validation Set Size: 430
Test Set Size: 432


# Step 3: PyTorch DataLoaders Initialization and Batch Sanity Check

In [3]:
# Define Hyperparameters
BATCH_SIZE = 32

# Instantiate PyTorch DataLoaders for parallel processing
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(" DataLoaders created successfully.")

# Sanity Check: Extract a single batch to verify shapes
data_iter = iter(train_loader)
images, labels = next(data_iter)

print("\n--- Sanity Check Output ---")
print(f"Batch Images Tensor Shape: {images.shape}")
# Expected output: [32, 3, 224, 224] -> [Batch_Size, Channels, Height, Width]
print(f"Batch Labels Tensor Shape: {labels.shape}")
# Expected output: [32] -> One scalar integer label per image
print("Step 2 pipeline completed without errors. Ready for model training!")

 DataLoaders created successfully.

--- Sanity Check Output ---
Batch Images Tensor Shape: torch.Size([32, 3, 224, 224])
Batch Labels Tensor Shape: torch.Size([32])
Step 2 pipeline completed without errors. Ready for model training!
